In [1]:
import sys
import os
import numpy as np
import scipy.linalg as la
from scipy.signal import cont2discrete
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter, FFMpegWriter
from mpl_toolkits.mplot3d import Axes3D 
import matplotlib as mpl

# Add project root to sys.path
project_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))
if project_root not in sys.path:
    sys.path.append(project_root)

import utils.config as cfg
from utils.quadcopter_model import quad_hover_linear_model, create_discrete_model, gaussian_dist, gaussian_dist_time_varying, gaussian_wind_direction, visualize_gaussian_samples, visualize_multi_task_gaussians, visualize_multi_task_gaussians_3D, plot_disturbances_time, animate_trajectories, plot_angles, plot_xyz, plot_controls_grid, plot_angles_mean, plot_xyz_mean, plot_controls_grid_mean, summarize_checks
from utils.simple_MLP import PolicyMLP, rollout_policy, traj_cost, train_nn_policy, meta_train_nn_policy, maml_adapt, evaluate_policy_cost, plot_maml_diagnostics, plot_adapted_losses, plot_adaptation_comparison, plot_loss_evolution

# Plotting parameters
plt.style.use('default')
plt.rcParams['figure.dpi'] = 400

# Enable LaTeX rendering
mpl.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "font.serif": ["Computer Modern Roman"], 
    "axes.unicode_minus": False 
})

# Consistent printing of numpy arrays
np.set_printoptions(precision=3, suppress=True)

In [ ]:
# Create and discretize model
A, B, C, D = quad_hover_linear_model()
# print("A =\n", A, "\n\nB =\n", B, "\n\nC =\n", C, "\n\nD =\n", D)

A_d, B_tilde_d, C_d, D_d = create_discrete_model(A, B, C, D)
B_d = B_tilde_d[:, :4]
# print("A_d =\n", A_d, "\n\nB_tilde_d =\n", B_tilde_d, "\n\nC_d =\n", C_d, "\n\nD_d =\n", D_d)

In [ ]:
# TRAINING POINTS
nb_points_training = 27

training_xyz = [
    [+5, +5, +5], [+5, +5,  0], [+5, +5, -5],
    [+5,  0, +5], [+5,  0,  0], [+5,  0, -5],
    [+5, -5, +5], [+5, -5,  0], [+5, -5, -5],
    
    [ 0, +5, +5], [ 0, +5,  0], [ 0, +5, -5],
    [ 0,  0, +5], [ 0,  0,  0], [ 0,  0, -5],
    [ 0, -5, +5], [ 0, -5,  0], [ 0, -5, -5],
    
    [-5, +5, +5], [-5, +5,  0], [-5, +5, -5],
    [-5,  0, +5], [-5,  0,  0], [-5,  0, -5],
    [-5, -5, +5], [-5, -5,  0], [-5, -5, -5]
]
x0_train = torch.zeros(nb_points_training, 12)

for i in range(nb_points_training):
    # x: [x, vx, y, vy, z, vz, roll, roll_rate, pitch, pitch_rate, yaw, yaw_rate]
    x0 = torch.tensor([training_xyz[i][0], 0.0, training_xyz[i][1], 0.0, training_xyz[i][2], 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]).float()
    x0_train[i, :] = x0

dx_train = torch.zeros(nb_points_training, cfg.T_STEPS, 1)
dy_train = torch.zeros(nb_points_training, cfg.T_STEPS, 1)
dz_train = -cfg.G * torch.ones_like(dx_train)
dist_train = torch.cat([dx_train, dy_train, dz_train], dim=-1)

print(x0_train.shape, dist_train.shape)

In [ ]:
def plot_loss_evolution(train_means, total_epochs, log_interval, filename="training_loss_evolution.png"):
    plt.figure(figsize=(7, 5))

    epochs_range = np.arange(log_interval, total_epochs + 1, log_interval)

    plt.plot(epochs_range, train_means, 'blue', alpha=0.8)
    plt.yscale("log")
    plt.xlabel("Epoch")
    plt.ylabel("Mean training loss")
    plt.grid(True, which="both", ls="--", lw=0.4)
    plt.tight_layout()
    plt.savefig(filename, dpi=300)
    plt.close()

In [ ]:
# TRAINING WITH NON LINEAR DYNAMICS
device = "cpu"
epochs = 1500
compute_mean_every = 100
lr = 1e-3

policy_non_linear, train_means_non_linear = train_nn_policy(
    A_d_nt=A_d, B_tilde_d_nt=B_tilde_d,
    init_points=x0_train, disturbances=dist_train,
    lr=lr, epochs=epochs,
    device=device, verbose_every=compute_mean_every, shuffle_each_epoch=True, use_linearized=False
)

In [ ]:
plot_loss_evolution(train_means_non_linear, epochs, compute_mean_every, filename="results_x0/loss_x0_non_linear.png")

In [ ]:
# TRAINING WITH LINEAR DYNAMICS
device = "cpu"
epochs = 1000
compute_mean_every = 100
lr = 1e-3

policy_linear, train_means_linear = train_nn_policy(
    A_d_nt=A_d, B_tilde_d_nt=B_tilde_d,
    init_points=x0_train, disturbances=dist_train,
    lr=lr, epochs=epochs,
    device=device, verbose_every=compute_mean_every, shuffle_each_epoch=True, use_linearized=True
)

In [ ]:
plot_loss_evolution(train_means_linear, epochs, compute_mean_every, filename="results_x0/loss_x0_linear.png")

In [ ]:
# TESTING POINTS
nb_points_testing = 10

# 20 random initial conditions in (-5,5)³
rng = np.random.default_rng(seed=42)
testing_points = (rng.random((nb_points_testing, 3)) * 10 - 5).tolist()

x0_test = torch.zeros(nb_points_testing, 12)

for i in range(nb_points_testing):
    # x: [x, vx, y, vy, z, vz, roll, roll_rate, pitch, pitch_rate, yaw, yaw_rate]
    x0 = torch.tensor([testing_points[i][0], 0.0, testing_points[i][1], 0.0, testing_points[i][2], 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]).float()
    x0_test[i, :] = x0

dx_test = torch.zeros(nb_points_testing, cfg.T_STEPS, 1)
dy_test = torch.zeros(nb_points_testing, cfg.T_STEPS, 1)
dz_test = -cfg.G * torch.ones_like(dx_test)
dist_test = torch.cat([dx_test, dy_test, dz_test], dim=-1)

print(x0_test.shape, dist_test.shape)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def plot_xyz_all_linear_comparison(X_lin, X_nonlin, filename):
    color_map = plt.get_cmap("twilight")

    # Check if tensors, convert to numpy if true
    if hasattr(X_lin, "detach"):  
        X_lin = X_lin.detach().cpu().numpy()
        X_nonlin = X_nonlin.detach().cpu().numpy()

    B, T, n = X_lin.shape
    t = np.arange(T) * cfg.T_D

    # Extract x, y, z (assuming indices 0, 2, 4)
    x_lin = X_lin[:, :, 0]
    y_lin = X_lin[:, :, 2]
    z_lin = X_lin[:, :, 4]

    x_nonlin = X_nonlin[:, :, 0]
    y_nonlin = X_nonlin[:, :, 2]
    z_nonlin = X_nonlin[:, :, 4]

    # Create 3 subplots for x, y, z
    fig, axes = plt.subplots(1, 3, figsize=(10, 4), sharey=True)
    
    # Color for plotting
    trajectory_color_lin = color_map(0.3)  # Linear trajectories color
    trajectory_color_nonlin = color_map(0.7)  # Nonlinear trajectories color

    # Plot for x coordinate
    for i in range(B):
        axes[0].plot(t, x_lin[i], color=trajectory_color_lin, alpha=0.7)  
        axes[0].plot(t, x_nonlin[i], color=trajectory_color_nonlin, alpha=0.7)  
        axes[0].fill_between(t, x_lin[i], x_nonlin[i], color='red', alpha=0.4)

    axes[0].set_xlabel("Time (s)")
    axes[0].set_ylim(-5, 5)
    axes[0].set_yticks(np.arange(-5, 6, 1)) 
    axes[0].set_ylabel("Position (m)")
    axes[0].set_title(r"$x$ Position")
    axes[0].grid(True)
    axes[0].set_xlim(0, 10)

    # Plot for y coordinate
    for i in range(B):
        axes[1].plot(t, y_lin[i], color=trajectory_color_lin, alpha=0.7)  
        axes[1].plot(t, y_nonlin[i], color=trajectory_color_nonlin, alpha=0.7)  
        axes[1].fill_between(t, y_lin[i], y_nonlin[i], color='red', alpha=0.4)

    axes[1].set_xlabel("Time (s)")
    axes[1].set_title(r"$y$ Position")
    axes[1].grid(True)
    axes[1].set_xlim(0, 10)

    # Plot for z coordinate
    for i in range(B):
        axes[2].plot(t, z_lin[i], color=trajectory_color_lin, alpha=0.7)  
        axes[2].plot(t, z_nonlin[i], color=trajectory_color_nonlin, alpha=0.7)  
        axes[2].fill_between(t, z_lin[i], z_nonlin[i], color='red', alpha=0.4)
    
    axes[2].plot([], [], color=trajectory_color_lin, alpha=0.7, label="Linearized dynamics")  
    axes[2].plot([], [], color=trajectory_color_nonlin, alpha=0.7, label="Nonlinear dynamics") 
    axes[2].fill_between([], [], [], color='red', alpha=0.4, label="Difference area") 


    axes[2].set_xlabel("Time (s)")
    axes[2].set_title(r"$z$ Position")
    axes[2].grid(True)
    axes[2].set_xlim(0, 10)

    # Adjust layout for better spacing
    plt.tight_layout()
    plt.legend(loc='upper right')

    # Save the figure
    plt.savefig(filename, dpi=300)
    plt.close()

In [ ]:
# TESTING on testing points
with torch.no_grad():
    X_testing_linear, U_testing_linear, mean_cost_testing_linear, std_cost_testing_linear, costs_testing_linear = evaluate_policy_cost(A_d, B_tilde_d, policy_linear, x0_test, dist_test, device=device, use_linearized=True)
    X_testing_non_linear, U_testing_non_linear, mean_cost_testing_non_linear, std_cost_testing_non_linear, costs_testing_non_linear = evaluate_policy_cost(A_d, B_tilde_d, policy_non_linear, x0_test, dist_test, device=device, use_linearized=False)


plot_xyz_all_linear_comparison(X_testing_linear, X_testing_non_linear, filename="results_x0/xyz_linear_vs_non_linear.png")

# plot_xyz_mean(X_testing_linear, filename="results_x0/xyz_linear_mean.png")
# plot_angles_mean(X_testing_linear, filename="results_x0/angles_linear_mean.png")
# plot_controls_grid_mean(U_testing_linear, filename="results_x0/controls_linear_mean.png")

# plot_xyz_mean(X_testing_non_linear, filename="results_x0/xyz_non_linear_mean.png")
# plot_angles_mean(X_testing_non_linear, filename="results_x0/angles_non_linear_mean.png")
# plot_controls_grid_mean(U_testing_non_linear, filename="results_x0/controls_non_linear_mean.png")